In [ ]:

from typing import TypedDict, Annotated
from operator import add

from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send


# =================================
# STATE
# =================================

class Task(BaseModel):
    description: str


class Plan(BaseModel):
    tasks: list[Task]


class AgentState(TypedDict):
    question: str
    tasks: list[Task]
    results: Annotated[list[str], add]
    final_answer: str


class WorkerState(TypedDict):
    task: Task


planner = llm.with_structured_output(Plan)


# =================================
# PLANNER
# =================================

def planner_node(state: AgentState):

    plan = planner.invoke(
        f"""
        Break this question into independent tasks.
        Create only tasks necessary to answer it.

        Question: {state['question']}
        """
    )

    return {
        "tasks": plan.tasks
    }


# =================================
# DISPATCHER
# =================================

def dispatch_tasks(state: AgentState):

    return [
        Send("worker", {"task": task})
        for task in state["tasks"]
    ]


# =================================
# WORKER
# =================================

def worker(state: WorkerState):

    task = state["task"]

    response = llm.invoke(
        f"Complete this task: {task.description}"
    )

    return {
        "results": [response.content]
    }


# =================================
# FINAL ANSWER
# =================================

def final_node(state: AgentState):

    response = llm.invoke(
        f"""
        Question: {state['question']}

        Results: {state['results']}

        Create a complete answer using these results.
        """
    )

    return {
        "final_answer": response.content
    }


# =================================
# BUILD GRAPH
# =================================

builder = StateGraph(AgentState)

builder.add_node("planner", planner_node)
builder.add_node("worker", worker)

builder.add_node("final", final_node)

builder.add_edge(START, "planner")

builder.add_conditional_edges(
    "planner",
    dispatch_tasks
)

builder.add_edge("worker", "final")
builder.add_edge("final", END)

graph = builder.compile()


# =================================
# RUN
# =================================

result = graph.invoke({
    "question": "Research LangChain, explain LangGraph, and compare them.",
    "results": []
})

print(result["final_answer"])
